In [23]:
import os
import pandas as pd

In [24]:
exp_root = "/media/yesindeed/DATADRIVE1/mount/remote_cse/experiments/med_vlm_benchmark/merged"

datasets = {
    "SLAKE": "/research/d5/gds/yzhong22/datasets/SLAKE/imgs",
    "PathVQA": "None",
    "VQA-RAD": "None",
    "Harvard-FairVLMed10k": "/research/d5/gds/yzhong22/datasets/Harvard-FairVLMed10k",
}

In [25]:
df_index = pd.read_csv(os.path.join(exp_root, "exp_status.csv"))
df_index = df_index.loc[df_index["trainable_module"].isin(["ucagent", "mdagent"])]
df_index

,model,task,dataset,model_type,trainable_module,path,have_eval_result,have_prediction,have_gpt_score,model_family,if_finished,error
104,Qwen2-VL,vqa,SLAKE,general,mdagent,vqa/SLAKE/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Inst...,1,1,0,Qwen,1,NaN
105,Qwen2-VL,vqa,SLAKE,general,ucagent,vqa/SLAKE/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Inst...,1,1,0,Qwen,1,NaN
106,Qwen25-VL,vqa,SLAKE,general,mdagent,vqa/SLAKE/Qwen25-VL/eval_seed0/Qwen2.5-VL-7B-I...,1,1,0,Qwen,1,NaN
107,Qwen25-VL,vqa,SLAKE,general,ucagent,vqa/SLAKE/Qwen25-VL/eval_seed0/Qwen2.5-VL-7B-I...,1,1,0,Qwen,1,NaN
108,Gemma3,vqa,SLAKE,general,mdagent,vqa/SLAKE/Gemma3/eval_seed0/gemma-3-4b-it/mdagent,1,1,0,Gemma,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
215,gemini-2.5-pro,vqa,OmniMedVQA,medical,ucagent,vqa/OmniMedVQA/gemini-2.5-pro/eval_seed0/none/...,0,0,0,Gemini,0,NaN
216,o3,vqa,OmniMedVQA,medical,mdagent,vqa/OmniMedVQA/o3/eval_seed0/none/mdagent,0,0,0,o-family,0,NaN
217,o3,vqa,OmniMedVQA,medical,ucagent,vqa/OmniMedVQA/o3/eval_seed0/none/ucagent,0,0,0,o-family,0,NaN
218,Lingshu,vqa,OmniMedVQA,medical,mdagent,vqa/OmniMedVQA/Lingshu/eval_seed0/Lingshu-7B/m...,1,1,0,Qwen,1,NaN


In [26]:
import re
from collections import defaultdict
import math
from typing import Optional, Iterable


def extract_choice_letter(response: str, choices: Iterable[str] = tuple("ABCDE")) -> Optional[str]:
    """
    Extract a single-letter multi-choice answer (e.g., A/B/C/D/E) from a model response.
    Supports both uppercase and lowercase letters.
    """
    if response is None:
        return None

    text = response.strip()
    if not text:
        return None

    # normalize whitespace
    t = re.sub(r"\s+", " ", text)

    choices_set = set(c.upper() for c in choices)
    if not choices_set:
        return None

    # Helper: validate and normalize
    def ok(letter: str) -> Optional[str]:
        u = letter.upper()
        return u if u in choices_set else None

    # 1) Explicit cues: "answer: c", "final answer is (b)"
    cue_patterns = [
        r"(?:final\s+answer|answer\s+is|answer|ans|choice|option)\s*[:\-]?\s*[\(\[]?\s*([A-Za-z])\s*[\)\]]?",
        r"(?:correct\s+answer\s+is|the\s+correct\s+answer\s+is)\s*[\(\[]?\s*([A-Za-z])\s*[\)\]]?",
    ]
    for pat in cue_patterns:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if m:
            letter = ok(m.group(1))
            if letter:
                return letter

    # 2) Parenthesized choices: "(b)"
    m = re.search(r"[\(\[]\s*([A-Za-z])\s*[\)\]]", t)
    if m:
        letter = ok(m.group(1))
        if letter:
            return letter

    # 3) Markdown bold: "**c**"
    m = re.search(r"\*\*\s*([A-Za-z])\s*\*\*", t)
    if m:
        letter = ok(m.group(1))
        if letter:
            return letter

    # 4) Fallback: standalone letter token
    m = re.search(r"\b([A-Za-z])\b", t)
    if m:
        letter = ok(m.group(1))
        if letter:
            return letter

    return None


def extract_option_answer(text):
    """
    从文本中提取 (A) xxx 形式的选项
    返回 dict: {"A": "...", "B": "...", ...}
    """
    options_dict = {}

    # 匹配 (A) xxx 直到下一个 (X) 或字符串结束
    pattern = r"\(([A-Z])\)\s*(.*?)(?=\s*\([A-Z]\)|$)"

    matches = re.findall(pattern, text, re.DOTALL)

    for key, value in matches:
        options_dict[key] = value.strip()

    return options_dict

In [27]:
text = """'A 19-year-old male high school wide receiver, who is right-hand dominant, reports recurrent subluxation of the right shoulder. Physical examination reveals a positive apprehension sign and a positive sulcus sign. A T2 coronal MRI image (Figure A) is provided. What is the most likely diagnosis?\nAnswer Choices: (A) Rotator cuff tear (B) HAGL lesion (C) ALPSA lesion (D) Bankart lesion (E) SLAP tear'"""


text = """What imaging modality is used to take this image?
Options:
(A) Microscopy
(B) Computed Tomography (CT) scan
(C) Thermography
(D) Positron Emission Tomography (PET) scan
Answer with the single letter corresponding to the best choice."""

extract_option_answer(text)

{'A': 'Microscopy',
 'B': 'Computed Tomography (CT) scan',
 'C': 'Thermography',
 'D': 'Positron Emission Tomography (PET) scan\nAnswer with the single letter corresponding to the best choice.'}

In [28]:
import re
import string


def normalize_text(text):
    """标准化文本"""
    text = text.lower().strip()
    text = re.sub(rf"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


def extract_options_from_question(question):
    """
    从问题中提取选项顺序
    支持:
    - lung,liver or heart
    - kidney or liver
    - MRI, CT or X-Ray
    """
    q = question.lower()

    # 取问号前的部分
    q = q.split("?")[0]

    # 找最后一个逗号或 or 开始的部分
    # 通过找最后一个动词后面的选项部分
    match = re.search(r",(.*)$", q)
    if match:
        option_part = match.group(1)
    else:
        match = re.search(r" (.+? or .+)", q)
        option_part = match.group(1) if match else ""

    # 用 or 统一分隔
    option_part = option_part.replace(" or ", ",")

    # 分割
    options = [o.strip() for o in option_part.split(",") if o.strip()]

    return options


def build_letter_mapping(options):
    """
    构建 A/B/C 映射
    """
    letters = list(string.ascii_uppercase)
    return {letters[i]: options[i] for i in range(len(options))}


def is_correct_vqa(question, gt_answer, model_answer):
    """
    判断模型回答是否正确
    - 自动处理 A/B/C 回答
    - 自动处理文本回答
    """
    options = extract_options_from_question(question)
    letter_map = build_letter_mapping(options)

    norm_gt = normalize_text(gt_answer)
    norm_model = normalize_text(model_answer)

    # 1️⃣ 如果模型直接回答文本
    if norm_model == norm_gt:
        return True

    # 2️⃣ 如果模型回答的是 A/B/C
    # 提取单个字母
    letter_match = re.match(r"\b([a-z])\b", norm_model)
    if letter_match:
        letter = letter_match.group(1).upper()
        if letter in letter_map:
            predicted_text = normalize_text(letter_map[letter])
            if predicted_text == norm_gt:
                return True

    return False

In [34]:
import tqdm
import json
import numpy as np


datasets_expect_closed_test_num = {
    "SLAKE": 416,
    "PathVQA": 3362,
    "VQA-RAD": 251,
    "Harvard-FairVLMed10k": 1996,
    "MedXpertQA": 400,
    "OmniMedVQA": 6186,
}

bootstrap_indices = {}

seed = 42
rng = np.random.default_rng(seed)
n_bootstraps = 1000

skip_list = []

acc = []
ci_low = []
ci_high = []

for i in tqdm.tqdm(range(len(df_index))):
    row = df_index.iloc[i]

    if row["if_finished"] != 1:
        continue

    # if row["trainable_module"] in ["mdagent", "ucagent"]:
    #     continue

    dataset = row["dataset"]
    path = row["path"]
    save_file = os.path.join(exp_root, path, "bootstrap_closed_meta.json")
    if any([x in path for x in skip_list]):
        continue

    # if os.path.exists(save_file):
    #     continue

    with open(os.path.join(exp_root, path, "predictions.json")) as fp:
        data = json.load(fp)

    accuracy_list = []

    for item in data:
        question_type = item["question_type"]

        if question_type == "open":
            continue
        else:
            answer = item["answer"].lower()
            pred = item["prediction"].lower()
            if dataset in ["MedXpertQA", "OmniMedVQA"]:
                question_type = "multi-choice"

            if question_type in ["yes/no", "closed"]:
                if answer in ["yes", "no"]:
                    accuracy = 1 if answer in pred else 0
                else:

                    accuracy = int((answer in pred) or is_correct_vqa(
                        item["qs"], answer, pred))
                    # print(
                    #     f"{dataset} - {row['model']} - {item['qs']} - {pred} (answer: {answer}) - {accuracy}")
            elif question_type == "multi-choice":
                answer_letter = extract_choice_letter(pred, tuple("ABCDEF"))
                if answer_letter is None:
                    accuracy = 0
                else:
                    if row["trainable_module"] in ["mdagent", "ucagent"]:
                        accuracy = (
                            1
                            if str.lower(answer_letter) == answer
                            or str.lower(extract_option_answer(item["qs"])[str.upper(answer)]) in pred
                            else 0
                        )
                    else:
                        accuracy = 1 if str.lower(
                            answer_letter) == answer else 0

            accuracy_list.append(accuracy)

    assert len(accuracy_list) == datasets_expect_closed_test_num[dataset]

    accuracy_list = np.array(accuracy_list)

    # generate boot indices for each dataset
    if dataset not in bootstrap_indices.keys():
        indices = []
        for _ in range(n_bootstraps):
            indices.append(
                rng.integers(
                    0, datasets_expect_closed_test_num[dataset], datasets_expect_closed_test_num[dataset])
            )
        bootstrap_indices[dataset] = indices

    bootstrap_results = {}
    for _ in range(n_bootstraps):
        sample_idx = bootstrap_indices[dataset][_]

        bootstrap_results[f"boots_{_}"] = {
            "idx": sample_idx.astype(int).tolist(),
            "accuracy": accuracy_list[sample_idx].astype(int).tolist(),
        }

    with open(save_file, "w") as fp:
        json.dump(bootstrap_results, fp)

100%|██████████| 116/116 [04:48<00:00,  2.49s/it]


In [81]:
item

{'image_path': '/tealab-data/rjin02/MedVLMBench/data/OmniMedVQA/OmniMedVQA/Images/Retinal OCT-C8/val/AMD/amd_val_1200.jpg',
 'prediction': 'B',
 'status': 'failed',
 'question_type': 'multi-choice',
 'qs': 'What are some distinguishing features of the abnormality captured in this image?',
 'answer': 'A',
 'prompt_template': '{}\nOptions:\n(A) Age-related Macular Degeneration (AMD) causes progressive damage to the macula, leading to vision loss in the center of the visual field.\n(B) Amblyopia: Amblyopia, also known as lazy eye, occurs when the brain and the eye do not work together correctly, leading to reduced vision in one eye. It is not directly related to the macula and is unrelated to age-related macular degeneration.\n(C) Diabetic retinopathy: Diabetic retinopathy is a complication of diabetes that affects the blood vessels in the retina. It can cause vision loss, but it is not specific to the macula and is different from age-related macular degeneration.\n(D) Glaucoma: Glaucoma 

In [79]:
extract_option_answer(item["qs"])

{}

In [80]:
item["qs"]

'What are some distinguishing features of the abnormality captured in this image?'

In [3]:
import json

with open(
    "/media/yesindeed/DATADRIVE1/mount/remote_cse/experiments/med_vlm_benchmark/merged/vqa/SLAKE/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Instruct/ucagent/predictions.json"
) as fp:
    data = json.load(fp)

data

[{'image_path': '/research/d5/gds/yzhong22/datasets/SLAKE/imgs/xmlab102/source.jpg',
  'question_type': 'open',
  'qs': 'What modality is used to take this image?',
  'answer': 'CT',
  'prediction': "The image is a computed tomography (CT) scan. CT scans use X-rays to create detailed images of the body's internal structures.",
  'trace': {'level1_expert1': '#Question Type: multi-choice\n#Reasoning: The image is a cross-sectional view of the thoracic region, showing the bony structures, soft tissues, and air-filled spaces. This type of imaging is characteristic of a computed tomography (CT) scan. CT scans are commonly used to evaluate the chest, abdomen, and pelvis for various conditions such as tumors, infections, and injuries.\n#Answer: CT',
   'error': {'question': 'What modality is used to take this image?',
    'prompt': '\n        [Core Identity] You are a professional and rigorous <MEDICAL FIELD> expert specializing in diagnostic imaging interpretation (<IMAGING MODALITIES>). You

In [6]:
for item in data:
    if item["question_type"] in ["yes/no"] and item["answer"] not in ["Yes", "No"]:
        print(item["qs"])

Which is the biggest in this image,lung,liver or heart?
Which is the biggest in this image,lung,liver or heart?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which organ is abnormal, heart or lung?
Which is bigger in this image, kidney or liver?
Which is the biggest in this image, lung,heart or liver?
Which is the biggest in this image, spleen,lung, or liver?
Which is bigger in this image, kidney or spleen?
Which is bigger in this image, liver or spleen?
Which is bigger in this image, liver or heart?
Which is bigger in this image, kidney or spleen?
Which is bigger in this image, liver or spleen?
Which side of lung is abnormal in this imag

In [12]:
import re
import string


def normalize_text(text):
    """标准化文本"""
    text = text.lower().strip()
    text = re.sub(rf"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


def extract_options_from_question(question):
    """
    从问题中提取选项顺序
    支持:
    - lung,liver or heart
    - kidney or liver
    - MRI, CT or X-Ray
    """
    q = question.lower()

    # 取问号前的部分
    q = q.split("?")[0]

    # 找最后一个逗号或 or 开始的部分
    # 通过找最后一个动词后面的选项部分
    match = re.search(r",(.*)$", q)
    if match:
        option_part = match.group(1)
    else:
        match = re.search(r" (.+? or .+)", q)
        option_part = match.group(1) if match else ""

    # 用 or 统一分隔
    option_part = option_part.replace(" or ", ",")

    # 分割
    options = [o.strip() for o in option_part.split(",") if o.strip()]

    return options


def build_letter_mapping(options):
    """
    构建 A/B/C 映射
    """
    letters = list(string.ascii_uppercase)
    return {letters[i]: options[i] for i in range(len(options))}


def is_correct_vqa(question, gt_answer, model_answer):
    """
    判断模型回答是否正确
    - 自动处理 A/B/C 回答
    - 自动处理文本回答
    """
    options = extract_options_from_question(question)
    letter_map = build_letter_mapping(options)

    norm_gt = normalize_text(gt_answer)
    norm_model = normalize_text(model_answer)

    # 1️⃣ 如果模型直接回答文本
    if norm_model == norm_gt:
        return True

    # 2️⃣ 如果模型回答的是 A/B/C
    # 提取单个字母
    letter_match = re.match(r"\b([a-z])\b", norm_model)
    if letter_match:
        letter = letter_match.group(1).upper()
        if letter in letter_map:
            predicted_text = normalize_text(letter_map[letter])
            if predicted_text == norm_gt:
                return True

    return False


question = "Which organ is abnormal, heart or lung?"
gt = "lung"

print(is_correct_vqa(question, gt, "liver"))  # ✅ True
print(is_correct_vqa(question, gt, "Liver"))  # ✅ True
print(is_correct_vqa(question, gt, "B"))  # ✅ True
print(is_correct_vqa(question, gt, "C"))  # ✅ True
print(is_correct_vqa(question, gt, "(C)"))  # ✅ True
print(is_correct_vqa(question, gt, "A"))  # ❌ False

False
False
True
False
False
False
